# 06 · Context-ablation timing: instruction flag on/off

Measures how long **one** context-ablation pass takes for a single QA, and
whether folding the generation system prompt into the ablation set (the
`instruction` flag) changes the wall-clock time.

**Design choices:**
- **In-process via `attribute_one`, not the HTTP API.** `POST /attribute` is
  hardwired to a `CachingProvider` and never passes `instruction`, so it can
  satisfy neither requirement here. We call `attribute_one()` (a drop-in for the
  engine `attribute_by_sentence`, in `tools/attribute_one.py`) directly instead —
  it's also the sandbox where ablation concurrency will be prototyped later.
- **No cache.** We use a bare `OpenAICompatProvider` (never wrapped in
  `CachingProvider`), so every ablation call hits NIM fresh. Timing is real,
  not a replay of `./.cache/logprobs.sqlite`.

Run order: setup → pick QA → knobs → **Cell A (no instruction)** → **Cell B
(with instruction)** → compare.

## Setup — non-caching provider

In [ ]:
import sys, time
from pathlib import Path

# Put the repo root (parent of this notebooks/ dir) on sys.path so `tools` and
# `dare` import regardless of cwd/kernel. `dare` is pip-installed (editable) so it
# would resolve anyway, but `tools` is NOT a package on the path — it's just a repo
# directory — so this line is what makes `from tools...` work.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dare.config import Settings
from dare.providers import OpenAICompatProvider
from dare.schema import load_corpus
from dare.prompts import SYNTHESIS_SYSTEM

# attribute_one is a drop-in for dare.attribution.attribute_by_sentence (same args,
# same {response, whole, units} return). It delegates to the engine today; it's the
# sandbox where ablation concurrency gets prototyped later, so the cells below don't
# change when that lands. See tools/attribute_one.py.
from tools.attribute_one import attribute_one

settings = Settings.from_env()  # loads repo-root .env regardless of cwd

def fresh_provider():
    """A bare provider with NO caching wrapper — every score_response() call goes
    to NIM. A fresh instance per run so nothing (even in-memory) carries over."""
    return OpenAICompatProvider(settings)

print(f"repo root  : {REPO_ROOT}")
print(f"model      : {settings.model}")
print(f"llm_url    : {settings.llm_url}")
print(f"call delay : {getattr(settings, 'api_call_delay', '?')}s between ablation calls (applies equally to both runs)")
print("caching    : OFF (bare OpenAICompatProvider)")

## Pick which QA to run

Set `QA_ID` to a record id, or leave it `None` and use `QA_INDEX`.

In [ ]:
CORPUS = "../data/raw_responses.jsonl"
QA_ID = None      # e.g. "qa-luongvh-0001"; if None, falls back to QA_INDEX
QA_INDEX = 0      # used only when QA_ID is None

records = load_corpus(CORPUS)

if QA_ID is not None:
    rec = next(r for r in records if r.id == QA_ID)
else:
    rec = records[QA_INDEX]

assert rec.attributable, f"record {rec.id!r} has no answer/sources to attribute"

print(f"id      : {rec.id}")
print(f"chunks  : {len(rec.chunks)}")
print(f"query   : {rec.query}")
print(f"answer  : {rec.answer[:300]}{'...' if len(rec.answer) > 300 else ''}")
print(f"\n(pick another: set QA_ID, or QA_INDEX in 0..{len(records) - 1})")

## Knobs

`NUM_ABLATIONS` drives the API-call count (dominant cost). `INSTRUCTION` is the
text folded in for Cell B — defaults to the real generation system prompt.

In [ ]:
NUM_ABLATIONS = 32          # ~this many sequential NIM calls per run
INSTRUCTION = SYNTHESIS_SYSTEM   # edit to test any instruction text

print(f"NUM_ABLATIONS = {NUM_ABLATIONS}")
print(f"INSTRUCTION   = {INSTRUCTION[:120]}...")

## Cell A — ablation WITHOUT the instruction flag

In [ ]:
t0 = time.perf_counter()
result_no_instr = attribute_one(
    rec.query,
    rec.answer,
    rec.chunks,
    num_ablations=NUM_ABLATIONS,
    provider=fresh_provider(),   # no cache
    settings=settings,
    # instruction omitted -> context-only ablation
)
elapsed_no_instr = time.perf_counter() - t0

print(f"WITHOUT instruction: {elapsed_no_instr:.2f}s  ({len(rec.chunks)} chunks, {NUM_ABLATIONS} ablations, {len(result_no_instr['units'])} units)")

## Cell B — ablation WITH the instruction flag

Same QA, same ablation count; the instruction is prepended into the context as
its own lane, so the ablation prompt is slightly longer per call.

In [ ]:
t0 = time.perf_counter()
result_with_instr = attribute_one(
    rec.query,
    rec.answer,
    rec.chunks,
    num_ablations=NUM_ABLATIONS,
    provider=fresh_provider(),   # no cache
    settings=settings,
    instruction=INSTRUCTION,     # folded in -> origin='instruction' lane
)
elapsed_with_instr = time.perf_counter() - t0

print(f"WITH instruction   : {elapsed_with_instr:.2f}s  ({len(rec.chunks)} chunks, {NUM_ABLATIONS} ablations, {len(result_with_instr['units'])} units)")

## Compare

In [ ]:
delta = elapsed_with_instr - elapsed_no_instr
pct = (delta / elapsed_no_instr * 100) if elapsed_no_instr else float('nan')

print(f"QA            : {rec.id}")
print(f"ablations     : {NUM_ABLATIONS}   chunks: {len(rec.chunks)}   cache: OFF")
print(f"without instr : {elapsed_no_instr:7.2f}s")
print(f"with instr    : {elapsed_with_instr:7.2f}s")
print(f"delta         : {delta:+7.2f}s  ({pct:+.1f}%)")

## Concurrency verification — serial vs concurrent must match

Runs the SAME QA twice with a **fresh, cache-off provider each time** so both
genuinely hit NIM live — the point is to verify the concurrent code path against
the real model, which a cache would short-circuit (you'd just replay the serial
run's logprobs and prove nothing):

1. **Serial** — `max_workers=1`, today's path.
2. **Concurrent** — `max_workers=MAX_WORKERS`, ablation calls fired in parallel.

Scoring is deterministic (`temperature 0`), so the attributions **must** be
identical — same masks, same logprobs, just dispatched differently. Any
difference means concurrency scrambled the mask↔logprob pairing and must not
ship. Only wall-clock should change.

On the `0.5s` `api_call_delay`: it still fires before each call, but the
per-thread sleeps overlap under concurrency, so it stops dominating the
concurrent run (and doesn't affect scores at all). Uncomment
`settings.api_call_delay = 0.0` in the serial cell for a cleaner speedup figure.

In [ ]:
MAX_WORKERS = 8   # concurrent ablation calls for the second run (start modest; watch for 429s)

# Optional: zero the throttle for a cleaner speedup number (doesn't affect scores).
# settings.api_call_delay = 0.0

# --- SERIAL run: max_workers=1 (today's path), fresh cache-off provider ---
t0 = time.perf_counter()
res_serial = attribute_one(
    rec.query,
    rec.answer,
    rec.chunks,
    num_ablations=NUM_ABLATIONS,
    provider=fresh_provider(),   # fresh, NO cache -> real live calls
    settings=settings,
    max_workers=1,               # serial baseline
)
elapsed_serial = time.perf_counter() - t0

scores_serial = [round(a.score, 4) for a in res_serial["whole"]]
print(f"SERIAL     : {elapsed_serial:6.2f}s   whole scores = {scores_serial}")

In [ ]:
# --- CONCURRENT run: max_workers > 1, fresh cache-off provider (real live calls) ---
t0 = time.perf_counter()
res_conc = attribute_one(
    rec.query,
    rec.answer,
    rec.chunks,
    num_ablations=NUM_ABLATIONS,
    provider=fresh_provider(),   # fresh, NO cache -> must hit NIM live to verify the concurrent path
    settings=settings,
    max_workers=MAX_WORKERS,     # concurrent ablation calls
)
elapsed_conc = time.perf_counter() - t0

scores_conc = [round(a.score, 4) for a in res_conc["whole"]]
print(f"CONCURRENT : {elapsed_conc:6.2f}s   whole scores = {scores_conc}   (max_workers={MAX_WORKERS})")

In [ ]:
import numpy as np

# whole-response scores, serial vs concurrent
s = np.array([a.score for a in res_serial["whole"]])
c = np.array([a.score for a in res_conc["whole"]])
whole_ok = s.shape == c.shape and np.allclose(s, c, atol=1e-6)

# every unit's scores must match too
unit_ok = all(
    np.allclose([x.score for x in us.attributions],
                [x.score for x in uc.attributions], atol=1e-6)
    for us, uc in zip(res_serial["units"], res_conc["units"])
)

print(f"whole max abs diff : {np.max(np.abs(s - c)):.2e}")
print(f"whole identical    : {whole_ok}")
print(f"units identical    : {unit_ok}")
print(f"speedup            : {elapsed_serial / elapsed_conc:.2f}x   ({elapsed_serial:.1f}s -> {elapsed_conc:.1f}s, max_workers={MAX_WORKERS})")
print()
print("PASS - concurrency matches serial, safe to ship" if (whole_ok and unit_ok)
      else "FAIL - results differ; concurrency scrambled pairing, do NOT ship")